## Notebook to demo how to create a feature class from a parquet file.

```
pip install duckdb
```

In [ ]:
import os

import arcpy
import duckdb
import pyarrow as pa

### Create an in-memory database and load the spatial extension.

In [ ]:
conn = duckdb.connect(":memory:")
_ = conn.execute("INSTALL spatial;LOAD spatial;")

### Read the parquet file.

In [ ]:
path = os.path.join("Z:", os.sep, "nff.prq", "*.parquet")

In [ ]:
_ = conn.sql(
    f"""
create or replace table tab as
select * exclude (SHAPE),ST_GeomFromWKB(SHAPE) as SHAPE
from read_parquet("{path}");
"""
)

In [ ]:
pdf = conn.sql("select PT_ID,PO_ID,ST_ASWKB(SHAPE) SHAPE from tab").df()

In [ ]:
sp_ref = arcpy.SpatialReference(3857)
metadata = {"esri.encoding": "WKB", "esri.sr_wkt": sp_ref.exportToString()}
pa_shape = pa.field("SHAPE", pa.binary(), nullable=False, metadata=metadata)

In [ ]:
feature_class_name = "NFF"
workspace = arcpy.env.scratchGDB

schema = pa.Schema.from_pandas(pdf)
shape_index = schema.get_field_index("SHAPE")
schema = schema.set(shape_index, pa_shape)
tab = pa.Table.from_pandas(pdf, schema=schema)

fc = os.path.join(workspace, feature_class_name)
arcpy.management.Delete(fc)
arcpy.management.CopyFeatures(tab, fc)